In [0]:
data = [
    (101,"Rahul",75000),
    (102,"Priya",85000),
    (103,"Amit",65000)
]

df = spark.createDataFrame(
    data,
    ["emp_id","name","salary"]
)

In [0]:
df.write.format("delta") \
.save("/tmp/employees_delta")

In [0]:
delta_df = spark.read.format("delta") \
.load("/tmp/employees_delta")

display(delta_df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:

df.write.format("delta") \
.saveAsTable("employees")

In [0]:
%sql select * from employees

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
%sql CREATE TABLE employees_delta
(
    emp_id INT,
    name STRING,
    salary INT
)
USING DELTA

In [0]:
%sql INSERT INTO employees_delta
VALUES
(101,'Rahul',75000),
(102,'Priya',85000)

num_affected_rows,num_inserted_rows
2,2


In [0]:
updates = [

(102,"Priya",90000),
(104,"Sneha",70000)

]

updates_df = spark.createDataFrame(
    updates,
    ["emp_id","name","salary"]
)

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(
    spark,
    "/tmp/employees_delta"
)

In [0]:
delta_table.alias("target") \
.merge(
    updates_df.alias("source"),
    "target.emp_id = source.emp_id"
) \
.whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",0) \
.load("/tmp/employees_delta")
display(df)

emp_id,name,salary
101,Rahul,75000
102,Priya,85000
103,Amit,65000


In [0]:
df = spark.read.format("delta") \
.option("versionAsOf",1) \
.load("/tmp/employees_delta")
display(df)

emp_id,name,salary
101,Rahul,75000
103,Amit,65000
102,Priya,90000
104,Sneha,70000


In [0]:
%sql
OPTIMIZE employees

ZORDER BY(salary)

path,metrics
abfss://unity-catalog-storage@dbstoragexilgy5hegv2j6.dfs.core.windows.net/7405618357181361/__unitystorage/catalogs/68e5bc19-968c-4d14-b2fd-1d6089169f27/tables/982a9823-633f-4b49-82e5-d07eacdb9609,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1308), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1781695951909, 1781695952413, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE priya.default.employees

path,metrics
abfss://unity-catalog-storage@dbstoragexilgy5hegv2j6.dfs.core.windows.net/7405618357181361/__unitystorage/catalogs/68e5bc19-968c-4d14-b2fd-1d6089169f27/tables/982a9823-633f-4b49-82e5-d07eacdb9609,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1781696059702, 1781696060379, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"
